**Q1: ANN with Vector Embeddings**



*   Preprocess the dataset to obtain vector representations for each conversational turn using a pre-trained embedding model (Glove and Sentence Transformers). You may calculate Glove Embeddings on a word level using pretrained model from Gensim 4 and average across words to get sentence level embeddings.
*   Implement a feedforward ANN architecture in PyTorch5, consisting of fully connected layers, nonlinear activation functions (e.g., ReLU), and
task-specific heads.


*   Define suitable loss functions (regression for emotion intensity/empathy,
classification for polarity). You may use any suitable loss functions for the task, for example - Mean Square Error (MSE)6 loss for regression and
Cross Entropy loss7 for classification.

*   Train your model on a CPU. You are free to experiment with different hy-
perparameters such as the number of hidden layers, hidden size, dropout,
and learning rate.
*    Evaluate the trained model on the development (dev) dataset and report performance using standard metrics (Mean Absolute Error8 for regression tasks and accuracy, precision, recall and F1 for classification).


*   Predict labels (Emotion, EmotionalPolarity, Empathy) for every sample
provided test dataset and save them in a CSV file for submission. The
CSV file should contain 4 columns - id, Emotion, EmotionalPolarity,
Empathy and each row should contain the predicted value for the corre-
sponding sample id.





In [1]:
#cell 1
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#cell 2
import pandas as pd
import os
import csv

base_dir = "/content/drive/MyDrive/P1_data"

train_path = os.path.join(base_dir, "trac2_CONVT_train.csv")
dev_path = os.path.join(base_dir, "trac2_CONVT_dev.csv")
test_path = os.path.join(base_dir, "trac2_CONVT_test.csv")

def robust_load(path):
    """Robust CSV loader to handle bad quotes and lines."""
    try:
        df = pd.read_csv(
            path,
            sep=",",
            engine="python",
            quoting=csv.QUOTE_MINIMAL,
            on_bad_lines="skip",
            encoding_errors="ignore"
        )
        print(f"Loaded {os.path.basename(path)} successfully (shape={df.shape})")
        return df
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None

train_df = robust_load(train_path)
dev_df = robust_load(dev_path)
test_df = robust_load(test_path)


for name, df in [("Train", train_df), ("Dev", dev_df), ("Test", test_df)]:
    print(f"\n--- {name} ---")
    if df is None:
        continue
    print("Columns:", df.columns.tolist())
    display(df.head(2))

expected_cols = ["id", "text", "Emotion", "EmotionalPolarity", "Empathy"]
missing = [c for c in expected_cols if c not in train_df.columns]
if missing:
    print(f"Missing expected columns in train: {missing}")
else:
    # Keep only relevant ones
    train_df = train_df[expected_cols]
    dev_df = dev_df[expected_cols]
    test_df = test_df[["id", "text"]]

print("\nCleaned datasets summary:")
print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

display(train_df.head(3))


Loaded trac2_CONVT_train.csv successfully (shape=(11090, 12))
Loaded trac2_CONVT_dev.csv successfully (shape=(965, 13))
Loaded trac2_CONVT_test.csv successfully (shape=(2294, 9))

--- Train ---
Columns: ['id', 'article_id', 'conversation_id', 'turn_id', 'speaker', 'text', 'person_id_1', 'person_id_2', 'Emotion', 'EmotionalPolarity', 'Empathy', 'SelfDisclosure']


,id,article_id,conversation_id,turn_id,speaker,text,person_id_1,person_id_2,Emotion,EmotionalPolarity,Empathy,SelfDisclosure
0,0,35,1,0,Person 1,what did you think about this article,p019,p012,1,1,1,1.0
1,1,35,1,1,Person 2,"It's definitely really sad to read, considerin...",p019,p012,3,2,4,2.0



--- Dev ---
Columns: ['id', 'article_id', 'conversation_id', 'turn_id', 'speaker_id', 'text', 'person_id', 'person_id_1', 'person_id_2', 'Emotion', 'EmotionalPolarity', 'Empathy', 'SelfDisclosure']


,id,article_id,conversation_id,turn_id,speaker_id,text,person_id,person_id_1,person_id_2,Emotion,EmotionalPolarity,Empathy,SelfDisclosure
0,1,8,68,1,38,Hello how are you?,38,38,75,1.0,1.0,1.0,NaN
1,2,8,68,2,75,Hello! Fine. How are you?,75,38,75,2.0,1.0,2.0,NaN



--- Test ---
Columns: ['id', 'article_id', 'conversation_id', 'turn_id', 'speaker_id', 'text', 'person_id', 'person_id_1', 'person_id_2']


,id,article_id,conversation_id,turn_id,speaker_id,text,person_id,person_id_1,person_id_2
0,1,27,393,1,188,"Yeah, I'm sorry but celebrity life doesn't int...",188,188,189
1,2,27,393,2,189,I'm pretty much the same. I like some of the m...,189,188,189



Cleaned datasets summary:
Train: (11090, 5)
Dev: (965, 5)
Test: (2294, 2)


,id,text,Emotion,EmotionalPolarity,Empathy
0,0,what did you think about this article,1,1,1
1,1,"It's definitely really sad to read, considerin...",3,2,4
2,2,I think it's super sad... they seem to never c...,4,2,5


In [3]:
#cell 3
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
print("punkt & punkt_tab downloaded successfully!.")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


punkt & punkt_tab downloaded successfully!.


In [4]:
#cell 4
!pip install numpy==1.26.4 gensim==4.3.2 --force-reinstall --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 21.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.2/63.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.4/121.4 kB 10.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is in

In [5]:
#cell 5
!pip install "scipy<1.12" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 1.9 MB/s eta 0:00:00
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 36.5 MB/s eta 0:00:00
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.2
    Uninstalling scipy-1.16.2:
      Successfully uninstalled scipy-1.16.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires scipy>=1.13, but you have scipy 1.11.4 which is incompatible

In [3]:
#cell 6
#Generate sentence embeddings using pre-trained GloVe
import gensim.downloader as api
import numpy as np
import nltk
from tqdm import tqdm
nltk.download('punkt')
from nltk.tokenize import word_tokenize

#Load pre-trained GloVe (100d)
print("Loading GloVe vectors (this may take a minute)...")
glove_model = api.load("glove-wiki-gigaword-100")
print(f"Loaded {len(glove_model.index_to_key)} word vectors.")

#Helper: compute mean vector for a sentence
def sentence_embedding(text, model, embedding_dim=100):
    if not isinstance(text, str) or not text.strip():
        return np.zeros(embedding_dim)
    tokens = word_tokenize(text.lower())
    valid_vecs = [model[word] for word in tokens if word in model]
    if not valid_vecs:
        return np.zeros(embedding_dim)
    return np.mean(valid_vecs, axis=0)

#Compute embeddings
def embed_dataframe(df, model):
    embs = []
    for text in tqdm(df["text"], desc="Embedding sentences"):
        embs.append(sentence_embedding(text, model))
    return np.vstack(embs)

print("\nGenerating embeddings for train/dev/test...")
train_embs = embed_dataframe(train_df, glove_model)
dev_embs = embed_dataframe(dev_df, glove_model)
test_embs = embed_dataframe(test_df, glove_model)

#Save embeddings for later use
os.makedirs("/content/wassa_embeddings", exist_ok=True)
np.save("/content/wassa_embeddings/train_glove.npy", train_embs)
np.save("/content/wassa_embeddings/dev_glove.npy", dev_embs)
np.save("/content/wassa_embeddings/test_glove.npy", test_embs)

print("\nEmbeddings created and saved:")
print("Train:", train_embs.shape)
print("Dev:", dev_embs.shape)
print("Test:", test_embs.shape)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


🔹 Loading GloVe vectors (this may take a minute)...
Loaded 400000 word vectors.

🔹 Generating embeddings for train/dev/test...


Embedding sentences: 100%|██████████| 2294/2294 [00:00<00:00, 4310.54it/s]


Embeddings created and saved:
Train: (11090, 100)
Dev: (965, 100)
Test: (2294, 100)


In [6]:
#Cell 7
#Train Feed-Forward ANN using GloVe embeddings

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, accuracy_score, precision_score, recall_score, f1_score


#Load embeddings from the folder where we saved them
base_path = "/content/wassa_embeddings"

train_path = os.path.join(base_path, "train_glove.npy")
dev_path   = os.path.join(base_path, "dev_glove.npy")
test_path  = os.path.join(base_path, "test_glove.npy")

#Safe loading
def safe_load(path):
    if os.path.exists(path):
        print(f"Loaded {os.path.basename(path)}")
        return np.load(path)
    else:
        print(f"Missing file: {path}")
        return None

#Try loading precomputed embeddings
train_embs = safe_load(train_path)
dev_embs   = safe_load(dev_path)
test_embs  = safe_load(test_path)

#Recompute embeddings if not found
if train_embs is None or dev_embs is None or test_embs is None:
    print("\nRecomputing GloVe embeddings because .npy files were missing...")
    import gensim.downloader as api
    from nltk.tokenize import word_tokenize
    from tqdm import tqdm
    import numpy as np

    glove_model = api.load("glove-wiki-gigaword-100")

    def sentence_embedding(text, model, embedding_dim=100):
        if not isinstance(text, str) or not text.strip():
            return np.zeros(embedding_dim)
        tokens = word_tokenize(text.lower())
        valid_vecs = [model[word] for word in tokens if word in model]
        if not valid_vecs:
            return np.zeros(embedding_dim)
        return np.mean(valid_vecs, axis=0)

    def embed_dataframe(df, model):
        embs = []
        for text in tqdm(df["text"], desc="Embedding sentences"):
            embs.append(sentence_embedding(text, model))
        return np.vstack(embs)


    train_embs = embed_dataframe(train_df, glove_model)
    dev_embs   = embed_dataframe(dev_df, glove_model)
    test_embs  = embed_dataframe(test_df, glove_model)

    os.makedirs(base_path, exist_ok=True)
    np.save(train_path, train_embs)
    np.save(dev_path, dev_embs)
    np.save(test_path, test_embs)
    print("Embeddings recomputed and saved successfully!")


print(f"\nEmbedding shapes:")
if train_embs is not None: print("Train:", train_embs.shape)
if dev_embs is not None:   print("Dev:", dev_embs.shape)
if test_embs is not None:  print("Test:", test_embs.shape)

#Prepare clean labels
def clean_polarity(series):
    """Fill NaN/invalid polarity labels with 2 (neutral) and clip to [1,3]."""
    series = series.fillna(2).astype(int)
    series = series.clip(1, 3)
    return series

train_df["EmotionalPolarity"] = clean_polarity(train_df["EmotionalPolarity"])
dev_df["EmotionalPolarity"]   = clean_polarity(dev_df["EmotionalPolarity"])

#Convert to tensors
X_train = torch.tensor(train_embs, dtype=torch.float32)
X_dev   = torch.tensor(dev_embs, dtype=torch.float32)

y_train_emotion  = torch.tensor(train_df["Emotion"].values, dtype=torch.float32)
y_train_polarity = torch.tensor(train_df["EmotionalPolarity"].values, dtype=torch.long) - 1   # 0-based
y_train_empathy  = torch.tensor(train_df["Empathy"].values, dtype=torch.float32)

y_dev_emotion  = torch.tensor(dev_df["Emotion"].values, dtype=torch.float32)
y_dev_polarity = torch.tensor(dev_df["EmotionalPolarity"].values, dtype=torch.long) - 1
y_dev_empathy  = torch.tensor(dev_df["Empathy"].values, dtype=torch.float32)

#Handle invalid polarity values
mask_train = (y_train_polarity >= 0) & (y_train_polarity < 3)
mask_dev   = (y_dev_polarity >= 0) & (y_dev_polarity < 3)
y_train_polarity = y_train_polarity[mask_train]
y_dev_polarity   = y_dev_polarity[mask_dev]
X_train = X_train[mask_train]
X_dev   = X_dev[mask_dev]
y_train_emotion = y_train_emotion[mask_train]
y_train_empathy = y_train_empathy[mask_train]
y_dev_emotion = y_dev_emotion[mask_dev]
y_dev_empathy = y_dev_empathy[mask_dev]

print("Labels prepared ->",
      f"Emotion:{y_train_emotion.shape}, Polarity:{y_train_polarity.shape}, Empathy:{y_train_empathy.shape}")


#Define Feed-Forward ANN
class MultiTaskANN(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.head_emotion  = nn.Linear(hidden_dim, 1)   # regression
        self.head_empathy  = nn.Linear(hidden_dim, 1)   # regression
        self.head_polarity = nn.Linear(hidden_dim, 3)   # classification

    def forward(self, x):
        h = self.shared(x)
        return (
            self.head_emotion(h).squeeze(1),
            self.head_polarity(h),
            self.head_empathy(h).squeeze(1)
        )


#Training setup
device = torch.device("cpu")

def batchify(X, *ys, batch_size=64):
    n = len(X)
    for i in range(0, n, batch_size):
        yield (X[i:i+batch_size],) + tuple(y[i:i+batch_size] for y in ys)


#Hyperparameter Tuning Loop (added for Q1)
from copy import deepcopy

param_grid = [
    {"hidden_dim": 64, "dropout": 0.2, "lr": 1e-3},
    {"hidden_dim": 128, "dropout": 0.3, "lr": 1e-3},
    {"hidden_dim": 256, "dropout": 0.4, "lr": 5e-4},
]

best_score = float("inf")
best_config = None
best_model_state = None

def evaluate_dev(model):
    model.eval()
    with torch.no_grad():
        pred_e, logits_p, pred_emp = model(X_dev)
        mae_emotion = mean_absolute_error(y_dev_emotion.numpy(), pred_e.numpy())
        mae_empathy = mean_absolute_error(y_dev_empathy.numpy(), pred_emp.numpy())
        preds_p = torch.argmax(logits_p, dim=1).numpy()
        f1_polarity = f1_score(y_dev_polarity.numpy(), preds_p, average="macro")
    return mae_emotion, mae_empathy, f1_polarity

print("\nStarting hyperparameter tuning...\n")

for cfg in param_grid:
    print(f"--- Training with hidden_dim={cfg['hidden_dim']}, dropout={cfg['dropout']}, lr={cfg['lr']} ---")

    model = MultiTaskANN(input_dim=100, hidden_dim=cfg["hidden_dim"], dropout=cfg["dropout"]).to(device)
    criterion_reg = nn.MSELoss()
    criterion_cls = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])

    n_epochs = 8
    batch_size = 64

    for epoch in range(1, n_epochs + 1):
        model.train()
        total_loss = 0.0
        for xb, y_e, y_p, y_emp in batchify(
            X_train, y_train_emotion, y_train_polarity, y_train_empathy, batch_size=batch_size
        ):
            optimizer.zero_grad()
            pred_e, logits_p, pred_emp = model(xb)
            loss = (
                criterion_reg(pred_e, y_e)
                + criterion_reg(pred_emp, y_emp)
                + criterion_cls(logits_p, y_p)
            )
            loss.backward()
            optimizer.step()
            total_loss += loss.item()


        mae_e, mae_emp, f1_p = evaluate_dev(model)
        score = mae_e + mae_emp - f1_p
        print(f"Epoch {epoch}: Loss={total_loss/len(X_train):.4f} | MAE_e={mae_e:.3f}, MAE_emp={mae_emp:.3f}, F1={f1_p:.3f}")

    if score < best_score:
        best_score = score
        best_config = cfg
        best_model_state = deepcopy(model.state_dict())

print("\nBest configuration found:")
print(best_config)


#Retrain final model using the best hyperparameters
print("\nRetraining final model using best configuration...")
print(best_config)

model = MultiTaskANN(
    input_dim=100,
    hidden_dim=best_config["hidden_dim"],
    dropout=best_config["dropout"]
).to(device)
model.load_state_dict(best_model_state)

criterion_reg = nn.MSELoss()
criterion_cls = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=best_config["lr"])

for epoch in range(1, 5):
    model.train()
    total_loss = 0.0
    for xb, y_e, y_p, y_emp in batchify(X_train, y_train_emotion, y_train_polarity, y_train_empathy, batch_size=64):
        optimizer.zero_grad()
        pred_e, logits_p, pred_emp = model(xb)
        loss = (
            criterion_reg(pred_e, y_e)
            + criterion_reg(pred_emp, y_emp)
            + criterion_cls(logits_p, y_p)
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Fine-tune Epoch {epoch}: Train Loss = {total_loss/len(X_train):.4f}")



#Evaluation on dev set
model.eval()
with torch.no_grad():
    pred_e, logits_p, pred_emp = model(X_dev)
    pred_p = torch.argmax(logits_p, dim=1)

#Regression → round for metrics
emotion_mae = mean_absolute_error(y_dev_emotion.numpy(), pred_e.numpy())
empathy_mae = mean_absolute_error(y_dev_empathy.numpy(), pred_emp.numpy())

#Classification
acc  = accuracy_score(y_dev_polarity.numpy(), pred_p.numpy())
prec = precision_score(y_dev_polarity.numpy(), pred_p.numpy(), average="macro", zero_division=0)
rec  = recall_score(y_dev_polarity.numpy(), pred_p.numpy(), average="macro", zero_division=0)
f1   = f1_score(y_dev_polarity.numpy(), pred_p.numpy(), average="macro", zero_division=0)

print("\nDEV PERFORMANCE:")
print(f"Emotion MAE : {emotion_mae:.4f}")
print(f"Empathy MAE : {empathy_mae:.4f}")
print(f"Polarity Acc : {acc:.4f} | Precision:{prec:.4f} | Recall:{rec:.4f} | F1:{f1:.4f}")

#Save model for later use
model_path = os.path.join(base_path, "glove_ann_model.pt")
torch.save(model.state_dict(), model_path)
print(f"\nModel saved to {model_path}")


Loaded train_glove.npy
Loaded dev_glove.npy
Loaded test_glove.npy

Embedding shapes:
Train: (11090, 100)
Dev: (965, 100)
Test: (2294, 100)
Labels prepared -> Emotion:torch.Size([11090]), Polarity:torch.Size([11090]), Empathy:torch.Size([11090])

🔍 Starting hyperparameter tuning...

--- Training with hidden_dim=64, dropout=0.2, lr=0.001 ---
Epoch 1: Loss=0.0517 | MAE_e=0.604, MAE_emp=0.848, F1=0.388
Epoch 2: Loss=0.0337 | MAE_e=0.575, MAE_emp=0.834, F1=0.603
Epoch 3: Loss=0.0313 | MAE_e=0.556, MAE_emp=0.831, F1=0.661
Epoch 4: Loss=0.0301 | MAE_e=0.550, MAE_emp=0.831, F1=0.700
Epoch 5: Loss=0.0295 | MAE_e=0.544, MAE_emp=0.838, F1=0.700
Epoch 6: Loss=0.0291 | MAE_e=0.548, MAE_emp=0.827, F1=0.713
Epoch 7: Loss=0.0287 | MAE_e=0.542, MAE_emp=0.828, F1=0.702
Epoch 8: Loss=0.0284 | MAE_e=0.542, MAE_emp=0.824, F1=0.729
--- Training with hidden_dim=128, dropout=0.3, lr=0.001 ---
Epoch 1: Loss=0.0452 | MAE_e=0.547, MAE_emp=0.841, F1=0.412
Epoch 2: Loss=0.0320 | MAE_e=0.537, MAE_emp=0.837, F1=0.69

In [7]:
#cell 8
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import csv


#Paths
base_path = "/content/wassa_embeddings"
model_path = os.path.join(base_path, "glove_ann_model.pt")
test_path = "/content/drive/MyDrive/P1_data/trac2_CONVT_test.csv"
test_embs_path = os.path.join(base_path, "test_glove.npy")

print("Using test file:", test_path)


#Model definition (same as training)
class SharedANN(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.head_emotion = nn.Linear(hidden_dim, 1)
        self.head_polarity = nn.Linear(hidden_dim, 3)
        self.head_empathy = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.shared(x)
        pred_e = self.head_emotion(x)
        logits_p = self.head_polarity(x)
        pred_emp = self.head_empathy(x)
        return pred_e, logits_p, pred_emp


#Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SharedANN()
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)
model.to(device)
model.eval()

print(f"Model loaded successfully from {model_path}")


#Load test embeddings + test data (robustly)
def robust_load_csv(path):
    """Load CSV while skipping malformed lines."""
    try:
        df = pd.read_csv(
            path,
            sep=",",
            engine="python",
            quoting=csv.QUOTE_MINIMAL,
            on_bad_lines="skip",
            encoding_errors="ignore"
        )
        print(f"Loaded {os.path.basename(path)} successfully (shape={df.shape})")
        return df
    except Exception as e:
        print(f"Error loading {path}: {e}")
        return None

test_df = robust_load_csv(test_path)
test_embs = np.load(test_embs_path)

print(f"Loaded test embeddings: {test_embs.shape}")


#Inference

X_test = torch.tensor(test_embs, dtype=torch.float32).to(device)

with torch.no_grad():
    pred_e, logits_p, pred_emp = model(X_test)
    pred_e = pred_e.cpu().numpy().flatten()
    pred_emp = pred_emp.cpu().numpy().flatten()
    pred_p = torch.argmax(logits_p, dim=1).cpu().numpy()


#Save predictions
output_df = pd.DataFrame({
    "id": test_df["id"],
    "pred_emotion": pred_e,
    "pred_polarity": pred_p,
    "pred_empathy": pred_emp
})

out_path = os.path.join(base_path, "submission.csv")
output_df.to_csv(out_path, index=False)

print(f"\Predictions saved to {out_path}")
display(output_df.head())


Using test file: /content/drive/MyDrive/P1_data/trac2_CONVT_test.csv
Model loaded successfully from /content/wassa_embeddings/glove_ann_model.pt
Loaded trac2_CONVT_test.csv successfully (shape=(2294, 9))
Loaded test embeddings: (2294, 100)
\Predictions saved to /content/wassa_embeddings/submission.csv


<>:105: SyntaxWarning: invalid escape sequence '\P'
<>:105: SyntaxWarning: invalid escape sequence '\P'
/tmp/ipython-input-659138154.py:105: SyntaxWarning: invalid escape sequence '\P'
  print(f"\Predictions saved to {out_path}")


,id,pred_emotion,pred_polarity,pred_empathy
0,1,2.135941,0,1.798337
1,2,2.590924,1,2.239986
2,3,2.302852,0,2.024153
3,4,2.220438,0,1.791775
4,5,2.447633,1,2.148640
